# **K-Nearest Neighbors (KNN) Classification on D&D Monsters**

In this notebook, we'll walk through performing k-nearest neighbors (KNN) classification using the **Dungeons & Dragons Monsters** dataset. We will follow these steps:

1. **Load and Inspect the Data:** Import and examine the dataset.
2. **Preprocess the Data:** Engineer the binary target and select features.
3. **Split the Data:** Divide the dataset into training and testing sets.
4. **Evaluate KNN Without Scaling:** Train and evaluate the KNN model on the original (unscaled) data.
5. **Scale the Data:** Apply feature scaling.
6. **Evaluate KNN With Scaling:** Re-run the model using the scaled data to see the improvement.
7. **Explore the Effect of Different k Values:** Analyze how varying the number of neighbors affects performance.

**Key question:** *Can we predict a D&D monster's size category from its ability scores and stats?*

This dataset is **perfect** for demonstrating why KNN needs scaling: ability scores range from 1–30, while HP ranges from 1–600+!

## Step 1: Load and Inspect the Data

We begin by importing the necessary libraries and loading the D&D Monsters dataset from TidyTuesday.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the D&D Monsters dataset from TidyTuesday
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-05-27/monsters.csv"
df = pd.read_csv(url)

# Display the first five rows of the dataset
print("Shape:", df.shape)
print("\nFirst five rows:")
df.head()

Shape: (330, 33)

First five rows:


,name,category,cr,size,type,descriptive_tags,alignment,ac,initiative,hp,...,wis_save,cha_save,skills,resistances,vulnerabilities,immunities,gear,senses,languages,full_text
0,Aboleth,Aboleth,10.00,Large,Aberration,NaN,Lawful Evil,17,7,150 (20d10 + 40),...,6,4,"History +12, Perception +10",NaN,NaN,NaN,NaN,Darkvision 120 ft.; Passive Perception 20,Deep Speech; telepathy 120 ft.,"Aboleth\nLarge Aberration, Lawful Evil\nAC 17\..."
1,Air Elemental,Air Elemental,5.00,Large,Elemental,NaN,Neutral,15,5,90 (12d10 + 24),...,0,-2,NaN,"Bludgeoning, Lightning, Piercing, Slashing",NaN,"Poison, Thunder; Exhaustion, Grappled, Paralyz...",NaN,Darkvision 60 ft.; Passive Perception 10,Primordial (Auran),"Air Elemental\nLarge Elemental, Neutral\nAC 15..."
2,Animated Armor,Animated Objects,1.00,Medium,Construct,NaN,Unaligned,18,2,33 (6d8 + 6),...,-4,-5,NaN,NaN,NaN,"Poison, Psychic; Charmed, Deafened, Exhaustion...",NaN,Blindsight 60 ft.; Passive Perception 6,NaN,"Animated Armor\nMedium Construct, Unaligned\nA..."
3,Animated Flying Sword,Animated Objects,0.25,Small,Construct,NaN,Unaligned,17,4,14 (4d6),...,-3,-5,NaN,NaN,NaN,"Poison, Psychic; Charmed, Deafened, Exhaustion...",NaN,Blindsight 60 ft.; Passive Perception 7,NaN,"Animated Flying Sword\nSmall Construct, Unalig..."
4,Animated Rug of Smothering,Animated Objects,2.00,Large,Construct,NaN,Unaligned,12,4,27 (5d10),...,-4,-5,NaN,NaN,NaN,"Poison, Psychic; Charmed, Deafened, Exhaustion...",NaN,Blindsight 60 ft.; Passive Perception 6,NaN,"Animated Rug of Smothering\nLarge Construct, U..."


In [ ]:
# Check available columns
print("Columns:")
print(df.columns.tolist())
print("\nSize distribution:")
print(df["size"].value_counts())

Columns:
['name', 'category', 'cr', 'size', 'type', 'descriptive_tags', 'alignment', 'ac', 'initiative', 'hp', 'hp_number', 'speed', 'speed_base_number', 'str', 'dex', 'con', 'int', 'wis', 'cha', 'str_save', 'dex_save', 'con_save', 'int_save', 'wis_save', 'cha_save', 'skills', 'resistances', 'vulnerabilities', 'immunities', 'gear', 'senses', 'languages', 'full_text']

Size distribution:
size
Large              107
Medium              90
Medium or Small     36
Huge                34
Tiny                25
Small               23
Gargantuan          15
Name: count, dtype: int64


## Step 2: Preprocess the Data

Before training our model, we need to:
- **Engineer a binary target:** `is_large` = 1 if size is "Large", "Huge", or "Gargantuan"; 0 otherwise.
- **Select numeric stat features:** ability scores (str, dex, con, int, wis, cha), hp_number, cr, speed_base_number, initiative.
- **Drop rows with NaN** in our selected columns.

Notice how different the feature scales are:
- Ability scores: 1–30
- HP: 1–600+
- CR: 0–30
- Speed: 0–120

This mismatch makes KNN distance calculations dominated by HP!

In [ ]:
# Engineer binary size target
large_sizes = ["Large", "Huge", "Gargantuan"]
df["is_large"] = df["size"].isin(large_sizes).astype(int)

print("Binary target distribution:")
print(df["is_large"].value_counts())

Binary target distribution:
is_large
0    174
1    156
Name: count, dtype: int64


In [ ]:
# Select numeric stat features
features = ["str", "dex", "con", "int", "wis", "cha", "hp_number", "cr", "speed_base_number", "initiative"]

# Drop rows with NaN in features or target
df_clean = df[features + ["is_large"]].dropna()
print(f"Clean dataset shape: {df_clean.shape}")

X = df_clean[features]
y = df_clean["is_large"]

# Preview the features — notice the different scales!
print("\nFeature ranges:")
print(X.describe().loc[["min", "max"]])

Clean dataset shape: (330, 11)

Feature ranges:
      str   dex   con   int   wis   cha  hp_number    cr  speed_base_number  \
min   1.0   1.0   8.0   1.0   3.0   1.0        1.0   0.0                5.0   
max  30.0  28.0  30.0  25.0  25.0  30.0      697.0  30.0               60.0   

     initiative  
min        -5.0  
max        20.0  


## Step 3: Split the Data

We split the dataset into training and testing subsets:
- **Training Set (80%):** Used to fit the k-nearest neighbors model.
- **Testing Set (20%):** Used to evaluate the model's performance.

A random state is set to ensure reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset: 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

## Step 4: Train KNN

KNN classifies a new data point by:
- Calculating the distance between the new data point and all points in the training set.
- Identifying the 'k' nearest neighbors.
- Assigning the class that is most common among these neighbors.

We initialize our KNN classifier with `k=5` and train it using training data.

In [ ]:
# Import KNeighborsClassifier

# Initialize the KNN classifier with k=5 neighbors

# Train the model using the training data


## Step 5: Evaluate KNN

To assess the performance of our KNN model, we will use:
- **Accuracy**
- **Confusion Matrix**
- **Classification Report**

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict the labels for the test dataset
y_pred = knn.predict(X_test)

# Calculate and print the accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy (k=5): {accuracy:.2f}")

# Generate the confusion matrix and display it using a heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('KNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Print the classification report with detailed metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Step 8: Explore the Effect of Different k Values

The performance of KNN can vary significantly with the choice of `k`. In this final step, we will:
- Vary `k` from 1 to 19 (odd numbers only).
- Train a KNN model for each `k` using the data.
- Plot the accuracy on the test set as a function of `k`.

This analysis will help illustrate the impact of different neighborhood sizes on the model's performance.

In [ ]:
# Define a range of k values to explore (odd numbers only)


# Loop through different values of k


# Plot accuracy vs. number of neighbors (k) for the scaled data
plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, marker='o')
plt.title('Accuracy vs. Number of Neighbors (k)')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Accuracy')
plt.xticks(k_values)
plt.show()

## Step 9: Scaling the Data

Since KNN is sensitive to the scale of the features, scaling can often lead to a substantial improvement in performance. Here, we use the StandardScaler to:
- Standardize the features so that they have a mean of 0 and a standard deviation of 1.
- Apply the transformation to both the training and testing data (using the training data to fit the scaler to avoid data leakage).

**Why is this especially important for D&D Monsters?**
- `hp_number` ranges from ~1 to 600+, while ability scores range from 1 to 30.
- Without scaling, HP dominates the distance calculation, making ability scores nearly irrelevant!

After scaling, we will re-run the KNN model.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Optional: Convert the scaled training data back to a DataFrame for inspection
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=features, index=X_train.index)
print("Scaled training features preview:")
print(X_train_scaled_df.head())

Scaled training features preview:
          str       dex       con       int       wis       cha  hp_number  \
75   0.535651 -1.267534  0.588504 -0.363235 -0.654350 -0.841733   0.368397   
184  1.136126 -0.954797  1.256263  1.016536 -0.317608  1.467878   0.766314   
17   1.136126  0.921628  0.588504 -0.190764  0.692616  0.313072   0.766314   
24  -0.515180  1.234365 -0.747013  0.326650  0.355875  0.148100  -0.650659   
180 -1.415893  0.296153 -0.524427 -0.363235 -0.654350 -0.346817  -0.602133   

           cr  speed_base_number  initiative  
75   0.049971          -0.123172   -1.085675  
184  0.726499           0.732547    0.150814  
17   1.064763           1.588266   -0.096484  
24  -0.753406           0.732547   -0.096484  
180 -0.711123           0.732547   -0.343781  


## Step 10: Evaluate KNN With Scaling

Now that we have scaled the features, we re-run the KNN model on the scaled data. We will use the same settings (k=5) to see how scaling affects the performance.

In [ ]:
# Initialize and train the KNN classifier with k=5 neighbors on the scaled data
knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)

# Predict the labels for the scaled test data
y_pred_scaled = knn_scaled.predict(X_test_scaled)

# Calculate and print the accuracy for the scaled data
accuracy_scaled = accuracy_score(y_test, y_pred_scaled)
print(f"Accuracy WITH scaling (k=5): {accuracy_scaled:.2f}")

# Generate and display the confusion matrix for the scaled data using a heatmap
cm_scaled = confusion_matrix(y_test, y_pred_scaled)
sns.heatmap(cm_scaled, annot=True, fmt='d', cmap='Blues')
plt.title('KNN Confusion Matrix (Scaled Data)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Print the classification report for the scaled data
print("\nClassification Report (Scaled Data):")
print(classification_report(y_test, y_pred_scaled))

## Step 11: Explore the Effect of Different k Values (Scaled Data)

The performance of KNN can vary significantly with the choice of `k`. In this final step, we will:
- Vary `k` from 1 to 19 (odd numbers only).
- Train a KNN model for each `k` using the scaled data.
- Plot the accuracy on the test set as a function of `k`.

This analysis will help illustrate the impact of different neighborhood sizes on the model's performance.

In [ ]:
# Define a range of k values to explore (odd numbers only)
k_values = range(1, 20, 2)
accuracies_scaled = []

# Loop through different values of k
for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train_scaled, y_train)
    y_temp_pred = knn_temp.predict(X_test_scaled)
    accuracies_scaled.append(accuracy_score(y_test, y_temp_pred))

# Plot accuracy vs. number of neighbors (k) for the scaled data
plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies_scaled, marker='o')
plt.title('Accuracy vs. Number of Neighbors (k) on Scaled Data')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Accuracy')
plt.xticks(k_values)
plt.show()